# Aula 14 — Uma Aplicação de Verdade

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

**Antes de começar, ligue a GPU.** No menu do Colab: *Ambiente de
execução* → *Alterar o tipo de ambiente de execução* → **T4 GPU** →
*Salvar*. É gratuito. Sem GPU o notebook funciona igual, só devagar.

Vamos construir um assistente que responde perguntas sobre este curso.
Dois modelos abertos, nenhuma chave de API.

## Parte A: Demonstração

### O material: as páginas deste curso

O arquivo `curso.txt` tem todas as páginas do GitBook, uma seção por
pedaço. Cada pedaço começa com `###` e o título da seção.

In [ ]:
import urllib.request

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/"
# Alternativa para testar offline:
# URL_DADOS = "../../data/"

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print(f"rodando em: {DISPOSITIVO}")

if URL_DADOS.startswith("http"):
    material = urllib.request.urlopen(URL_DADOS + "curso.txt").read().decode("utf-8")
else:
    material = open(URL_DADOS + "curso.txt", encoding="utf-8").read()

pedacos = [("### " + p).strip() for p in material.split("### ") if p.strip()]

tamanhos = sorted(len(p) for p in pedacos)
print(f"{len(pedacos)} pedaços")
print(f"menor {tamanhos[0]}, mediana {tamanhos[len(tamanhos) // 2]}, "
      f"maior {tamanhos[-1]} caracteres")
print()
print(pedacos[10][:400])

### Transformar texto em vetor

Passa o texto pelo modelo, pega os vetores de todos os tokens e tira a
média. É o **embutimento de frase**.

$$v = \frac{1}{n}\sum_{i=1}^{n} h_i$$

O modelo é o `multilingual-e5-small`, com 118 milhões de pesos. Ele pede
o prefixo `passage:` nos documentos e `query:` nas perguntas.

In [ ]:
BUSCADOR = "intfloat/multilingual-e5-small"

tok_busca = AutoTokenizer.from_pretrained(BUSCADOR)
mod_busca = AutoModel.from_pretrained(BUSCADOR).to(DISPOSITIVO)
mod_busca.eval()


@torch.no_grad()
def embutir(textos, prefixo, lote=32):
    saidas = []
    for inicio in range(0, len(textos), lote):
        entradas = tok_busca([prefixo + t for t in textos[inicio:inicio + lote]],
                             padding=True, truncation=True, max_length=256,
                             return_tensors="pt").to(DISPOSITIVO)
        estados = mod_busca(**entradas).last_hidden_state
        mascara = entradas["attention_mask"].unsqueeze(-1).float()
        media = (estados * mascara).sum(1) / mascara.sum(1)
        saidas.append(F.normalize(media, dim=-1).cpu())
    return torch.cat(saidas)


vetores = embutir(pedacos, "passage: ")
print(f"{tuple(vetores.shape)}: {vetores.shape[0]} pedaços, "
      f"{vetores.shape[1]} números cada")
print(f"tamanho de cada vetor: {vetores[0].norm():.2f}")

### Parecença sem palavra em comum

Com vetores de tamanho 1, o produto escalar da Aula 10 vira a
**similaridade do cosseno**: −1 para opostos, 1 para idênticos.

In [ ]:
frases = [
    "O gato dormiu no sofá a tarde inteira.",
    "O cachorro cochilou no tapete da sala.",
    "A rede neural aprendeu a separar as fotos.",
]
v = embutir(frases, "query: ")

print(f"gato x cachorro:  {v[0] @ v[1]:.3f}  (nenhuma palavra em comum)")
print(f"gato x rede:      {v[0] @ v[2]:.3f}")
print(f"cachorro x rede:  {v[1] @ v[2]:.3f}")

### Buscar

Uma multiplicação de matriz e um `topk`. Com milhões de pedaços isso
viraria uma base vetorial; com 162, uma matriz resolve.

In [ ]:
def buscar(pergunta, quantos=3):
    vetor = embutir([pergunta], "query: ")[0]
    melhores = torch.topk(vetores @ vetor, quantos)
    return melhores.indices.tolist(), melhores.values.tolist()


achados, notas = buscar("O que é a máscara causal?", 4)
for indice, nota in zip(achados, notas):
    print(f"{nota:.3f}  {pedacos[indice].splitlines()[0][4:]}")

### O modelo que responde

É o mesmo da Aula 13. Nada nele foi treinado com este material.

In [ ]:
GERADOR = "Qwen/Qwen2.5-0.5B-Instruct"

tok_ger = AutoTokenizer.from_pretrained(GERADOR)
mod_ger = AutoModelForCausalLM.from_pretrained(GERADOR).to(DISPOSITIVO)
mod_ger.eval()


def responder(texto, quantos=60):
    mensagens = [{"role": "user", "content": texto}]
    molde = tok_ger.apply_chat_template(mensagens, tokenize=False,
                                        add_generation_prompt=True)
    entradas = tok_ger(molde, return_tensors="pt").to(DISPOSITIVO)
    with torch.no_grad():
        saida = mod_ger.generate(**entradas, max_new_tokens=quantos,
                                 do_sample=False)
    novos = saida[0][entradas.input_ids.shape[1]:]
    return " ".join(tok_ger.decode(novos, skip_special_tokens=True).split())


print(responder("O que significa a sigla BPE?", 45))

Isso não existe. A resposta certa está escrita na Aula 9, que ele nunca
leu.

### O prompt aumentado

Cola os trechos buscados antes da pergunta. Três decisões, todas vindas
de tentar: mandar usar só o texto, cortar os trechos, e limitar o
formato da resposta.

In [ ]:
def perguntar(pergunta, quantos_trechos=3, corte=900):
    achados, _ = buscar(pergunta, quantos_trechos)
    contexto = "\n\n".join(pedacos[i][:corte] for i in achados)
    prompt = (f"Use apenas o texto abaixo para responder.\n\n{contexto}\n\n"
              f"Pergunta: {pergunta}\n"
              "Responda em uma frase curta, só com o que está no texto.")
    return responder(prompt, 45)


for pergunta in ["O que significa a sigla BPE?",
                 "O que é a máscara causal?"]:
    print(f"P: {pergunta}")
    print(f"   sem RAG: {responder(pergunta, 40)}")
    print(f"   com RAG: {perguntar(pergunta)}")
    print()

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: conhecendo os pedaços

Rode e observe: de quais aulas vêm os pedaços, e quantos de cada uma?

In [ ]:
import collections

por_aula = collections.Counter(p.splitlines()[0][4:].split(" > ")[0]
                               for p in pedacos)
for aula, quantos in por_aula.most_common(8):
    print(f"{quantos:>3}  {aula}")

In [ ]:
if len(pedacos) > 100:
    print(f"✅ {len(pedacos)} pedaços, de {len(por_aula)} páginas diferentes.")
    print("   Cada pedaço é uma seção, e o título dele entra na busca.")
else:
    print("❌ Confira se a célula que baixa o material rodou.")

### Exercício 2: o vetor de uma frase

Complete `meu_vetor`, o embutimento da sua frase. Use `embutir` com o
prefixo `"query: "`, que é o que esse modelo pede para perguntas.

In [ ]:
minha_frase = "Como uma rede neural aprende a reconhecer imagens?"

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"tamanho do vetor: {len(meu_vetor)} números")
print(f"norma: {meu_vetor.norm():.3f}")
print(f"primeiros cinco: {meu_vetor[:5]}")

In [ ]:
if len(meu_vetor) == 384 and abs(float(meu_vetor.norm()) - 1.0) < 0.01:
    print("✅ 384 números, com tamanho 1.")
    print("   Normalizado, o produto escalar já é a similaridade do cosseno.")
else:
    print("❌ Confira: embutir([sua_frase], 'query: ')[0].")

### Exercício 3: comparando duas frases

Calcule `minha_parecenca`: a similaridade entre `frase_a` e `frase_b`.
Repare que elas não têm palavra em comum.

In [ ]:
frase_a = "A cafeteria abre às sete da manhã."
frase_b = "O café fica pronto logo cedo."
frase_c = "O modelo treinou por quarenta minutos."

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"a x b (mesmo assunto):  {vetores_teste[0] @ vetores_teste[1]:.3f}")
print(f"a x c (outro assunto):  {vetores_teste[0] @ vetores_teste[2]:.3f}")

In [ ]:
if minha_parecenca > float(vetores_teste[0] @ vetores_teste[2]):
    print("✅ As duas do mesmo assunto ficaram mais perto.")
    print("   Nenhuma palavra em comum, e a busca por palavra não veria isso.")
else:
    print("❌ Confira se você embutiu as três frases juntas.")

### Exercício 4: a sua função de busca

Escreva `minha_busca(pergunta, quantos)`, que devolve os títulos dos
pedaços mais parecidos. Três passos: embutir a pergunta, multiplicar
pela matriz, pegar os maiores.

In [ ]:
def minha_busca(pergunta, quantos=3):
    # SEU CODIGO AQUI
    return []

In [ ]:
# SEU CODIGO AQUI

In [ ]:
for titulo in minha_busca("Por que a temperatura muda o texto gerado?"):
    print(" ", titulo)

In [ ]:
resultado = minha_busca("Como o Prophet lida com feriados?", 3)
if resultado and "Prophet" in resultado[0]:
    print("✅ A busca trouxe a seção certa da Aula 5.")
else:
    print(f"❌ Esperava algo da Aula 5 e veio: {resultado}")

### Exercício 5: o prompt aumentado

Escreva `meu_prompt(pergunta)`, que monta o texto com os trechos
buscados. Três partes: a instrução, os trechos, e a pergunta.

In [ ]:
def meu_prompt(pergunta, quantos_trechos=3, corte=900):
    achados, _ = buscar(pergunta, quantos_trechos)
    # SEU CODIGO AQUI
    return pergunta

In [ ]:
# SEU CODIGO AQUI

In [ ]:
minha_pergunta = "Para que serve o pooling numa rede convolucional?"
prompt_montado = meu_prompt(minha_pergunta)

print(f"o prompt ficou com {len(prompt_montado)} caracteres")
print(f"({len(tok_ger(prompt_montado).input_ids)} tokens)")
print()
print(responder(prompt_montado, 45))

In [ ]:
if len(prompt_montado) > 1000 and minha_pergunta in prompt_montado:
    print("✅ Prompt montado com os trechos e a pergunta dentro.")
else:
    print("❌ Confira: instrução, depois os trechos, depois a pergunta.")

### Exercício 6: avaliando

Rode e observe. Quatro perguntas cuja resposta você conhece, com e sem
RAG. Conte quantas cada modo acerta.

Isso leva alguns minutos sem GPU. Rode uma vez e espere.

In [ ]:
perguntas = [
    ("O que significa a sigla BPE?", "codificação por pares de bytes"),
    ("Quantos pesos tem o modelo de linguagem do curso?", "787.584"),
    ("Quantas categorias tem o conjunto Fashion-MNIST?", "dez"),
    ("O que é a máscara causal?", "zera o que aponta para o futuro"),
]

for pergunta, certo in perguntas:
    print(f"P: {pergunta}")
    print(f"   certo:   {certo}")
    print(f"   sem RAG: {responder(pergunta, 35)[:110]}")
    print(f"   com RAG: {perguntar(pergunta)[:110]}")
    print()

In [ ]:
print("Conte com um colega: quantas cada modo acertou?")
print("Uma demonstração que funciona não prova nada. O conjunto de")
print("perguntas com resposta conhecida é o mínimo de um sistema sério.")

### Exercício 7: desafio, quebre a busca

Ache uma pergunta sobre o curso em que a busca traz o trecho **errado**.
Guarde em `pergunta_dificil`.

Dica: perguntas com palavras que aparecem em várias aulas, ou perguntas
que misturam dois assuntos.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
# SEU CODIGO AQUI

In [ ]:
achados_dificeis, notas_dificeis = buscar(pergunta_dificil, 4)
print(f"P: {pergunta_dificil}")
for indice, nota in zip(achados_dificeis, notas_dificeis):
    print(f"   {nota:.3f}  {pedacos[indice].splitlines()[0][4:]}")
print()
print("resposta:", perguntar(pergunta_dificil))

In [ ]:
print("Toda aula tem um 'erro do dia', então a pergunta é ambígua e a busca")
print("não tem como escolher. Numa aplicação de verdade, isso se resolve")
print("filtrando por documento antes de buscar, ou pedindo que o usuário")
print("diga de qual aula está falando.")

Agora, em texto: escolha um documento do seu trabalho e descreva, em
quatro frases, como você montaria um assistente sobre ele. Diga também
quais cinco perguntas você usaria para avaliar se ele funciona. Edite
esta célula (duplo clique nela) e escreva sua resposta no lugar deste
parágrafo.